# dataclass-training-args — ex2: diagnose mutable-default trap + repair with default_factory

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dataclass-training-args`. Running the final beacon cell reports progress against the `Config: @dataclass training args` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: @dataclass training args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataclass-training-args`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataclass-training-args"
DD_SUBTOPIC = "Config: @dataclass training args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `@dataclass` mutable-default trap — quick refresher

Python dataclasses REJECT a mutable default on a field:

```python
@dataclass
class Args:
    layer_dims: list = [256, 128]   # ValueError at class-creation
```

**The fix:** `field(default_factory=lambda: [256, 128])`. The factory runs once per *instance*, so two `Args()` calls get two independent lists.

**Why this matters for training args.** If you accidentally share a list across runs (e.g. by patching defaults at module load), mutating it in one run silently corrupts the other. The `default_factory` rule prevents this at class-definition time.

**`frozen=True` is the heavier hammer.** Forbids ALL attribute assignment after `__init__`. Useful for args that should be immutable for the entire run; not useful for args you mutate from a sweep agent.

### Exercise 2 — diagnose mutable-default trap + repair with default_factory

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze a broken @dataclass definition to detect the mutable-default pitfall and return a repaired class that uses `field(default_factory=...)` for the list default.
> Keywords: dataclass, default-factory, mutable-default, frozen
> ```

**KCs targeted:** `dataclass-default-factory`, `dataclass-frozen-immutability`

Implement `ex2_build_safe_args_class()`. Build (and RETURN) a dataclass `SafeArgs` that:

1. Has fields:
   - `lr: float = 1e-3`
   - `batch_size: int = 32`
   - `layer_dims: list[int]` — defaults to `[256, 128]` BUT via `field(default_factory=...)`, so two instances get independent lists.
2. Is NOT frozen (the test will mutate `lr` on an instance).
3. Returns the class object itself, not an instance.

The test will verify:
- Two `SafeArgs()` instances have `layer_dims` lists with DIFFERENT identities (`is not`).
- Mutating one instance's `layer_dims.append(64)` does NOT affect the other instance's `layer_dims`.
- Mutating `lr` on an instance succeeds (proves it is not frozen).
- A naive class definition `layer_dims: list = [256, 128]` would have raised `ValueError` at class-creation time — your repaired version doesn't.

**Hint:** import `field` from `dataclasses`.

In [ ]:
def ex2_build_safe_args_class():
    """Return a dataclass that avoids the mutable-default trap."""
    raise NotImplementedError()


def _test_ex2():
    from dataclasses import is_dataclass, fields, FrozenInstanceError

    SafeArgs = ex2_build_safe_args_class()

    # Class-level checks.
    assert is_dataclass(SafeArgs), 'must be a dataclass'
    field_names = {f.name for f in fields(SafeArgs)}
    assert field_names == {'lr', 'batch_size', 'layer_dims'}, (
        f'expected fields lr/batch_size/layer_dims, got {field_names}'
    )

    # Default-factory test: two instances → two independent lists.
    a = SafeArgs()
    b = SafeArgs()
    assert a.layer_dims == [256, 128], f'default layer_dims wrong: {a.layer_dims}'
    assert b.layer_dims == [256, 128]
    assert a.layer_dims is not b.layer_dims, (
        'two instances share the same list — you used `= [256, 128]` directly '
        'instead of `field(default_factory=...)`'
    )

    # Mutating one must not affect the other.
    a.layer_dims.append(64)
    assert a.layer_dims == [256, 128, 64]
    assert b.layer_dims == [256, 128], 'shared list leaked across instances'

    # Not frozen: lr is assignable.
    a.lr = 5e-4
    assert a.lr == 5e-4

    # Numeric defaults intact.
    assert SafeArgs().lr == 1e-3
    assert SafeArgs().batch_size == 32
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
from dataclasses import dataclass, field

def ex2_build_safe_args_class():
    @dataclass
    class SafeArgs:
        lr: float = 1e-3
        batch_size: int = 32
        layer_dims: list = field(default_factory=lambda: [256, 128])
    return SafeArgs
```

**Why `default_factory` exists.** Without it, the default value would be evaluated once at class-creation time and SHARED across every instance — classic Python mutable-default-argument bug. Dataclasses detect this and raise `ValueError` rather than let you ship a footgun.

**Lambda vs `list`.** `field(default_factory=list)` gives `[]`. For a non-empty default, pass a lambda returning the literal: `lambda: [256, 128]`.

**`frozen=True` trade-off.** Freezing prevents accidental mutation (good) but also prevents sweep agents from doing `args.lr = sweep_lr` (bad). ARENA convention: NOT frozen — training args are mutable for sweep overrides.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()